In [1]:
!pip install scikit-learn
!pip install ddgs
!pip install pillow

In [2]:

import os
import requests
import time
from PIL import Image
from io import BytesIO
from ddgs import DDGS
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader, random_split

In [3]:
CLASSES = ["car", "truck", "bicycle", "pedestrian"]
DATA_DIR = "data"

IMAGES_PER_CLASS = 20   # increase later for better accuracy
BATCH_SIZE = 16
EPOCHS = 10
LR = 0.001

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)




Device: cpu


In [4]:
for cls in CLASSES:
    os.makedirs(f"{DATA_DIR}/{cls}", exist_ok=True)

In [5]:
def download_images(query, folder, max_images=500):
    with DDGS() as ddgs:
        results = ddgs.images(query, max_results=max_images)

        count = 0
        for r in results:
            try:
                url = r["image"]
                response = requests.get(url, timeout=5)

                img = Image.open(BytesIO(response.content)).convert("RGB")
                img = img.resize((224, 224))

                img.save(f"{folder}/{query}_{count}.jpg")

                count += 1
                time.sleep(0.2)  # IMPORTANT

                if count >= max_images:
                    break

            except:
                pass


for cls in CLASSES:
    print("Downloading:", cls)
    download_images(cls, f"{DATA_DIR}/{cls}", IMAGES_PER_CLASS)

print("Dataset ready!")

Downloading: car
Downloading: truck
Downloading: bicycle
Downloading: pedestrian
Dataset ready!


In [19]:
# ──────────────────────────────────────────────────────────────
# 1.  BUILD & LOAD MODEL
# CLASS_NAMES = ["car", "truck", "bicycle", "pedestrian"]
#Classes: {'bicycle': 0, 'car': 1, 'pedestrian': 2, 'truck': 3}
CLASS_NAMES = ['bicycle', 'car', 'pedestrian', 'truck']
NUM_CLASSES = len(CLASS_NAMES)
def load_model(model_path: str, num_classes: int = NUM_CLASSES):
    if os.path.exists(model_path):
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

        state = torch.load(model_path, map_location=DEVICE)
        model.load_state_dict(state)
        print(f"[OK] Loaded weights from '{model_path}'")
    else:
        print(f"[WARNING] '{model_path}' not found — using ImageNet pretrained model")
        model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)


    model = model.to(DEVICE)
    model.eval()
    return model



In [20]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(),
    transforms.ToTensor(),


])

In [22]:

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print("Classes:", dataset.class_to_idx)




Classes: {'bicycle': 0, 'car': 1, 'pedestrian': 2, 'truck': 3}


In [23]:
# Load pretrained model
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(DEVICE)


In [24]:
# Step 1: Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Step 2: Unfreeze last block (layer4)
for param in model.layer4.parameters():
    param.requires_grad = True

# Step 3: Ensure FC layer is trainable
for param in model.fc.parameters():
    param.requires_grad = True

In [25]:
#Loss + optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': 1e-4}
])

In [26]:
#Training loop
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # ✅ Accuracy calculation
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Accuracy: {train_acc:.2f}%")


Epoch 1, Loss: 11.4386, Accuracy: 28.12%
Epoch 2, Loss: 9.5935, Accuracy: 49.22%
Epoch 3, Loss: 7.6795, Accuracy: 72.66%
Epoch 4, Loss: 6.2720, Accuracy: 89.06%
Epoch 5, Loss: 5.4778, Accuracy: 89.06%
Epoch 6, Loss: 4.6385, Accuracy: 91.41%
Epoch 7, Loss: 4.1348, Accuracy: 94.53%
Epoch 8, Loss: 3.6516, Accuracy: 95.31%
Epoch 9, Loss: 3.2274, Accuracy: 96.09%
Epoch 10, Loss: 2.9148, Accuracy: 95.31%


In [28]:
#Evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 100.0


In [31]:
print(preds[:10])
print(labels[:10])

tensor([0])
tensor([0])


In [32]:
#Save model
torch.save(model.state_dict(), "best_model.pth")
print("Model saved!")



Model saved!


In [33]:
import torch.nn.functional as F

def predict_image(image_path, model, transform, class_names):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        probs = F.softmax(outputs, dim=1)

    for i, p in enumerate(probs[0]):
        print(class_names[i], float(p))

    _, pred = torch.max(probs, 1)
    return class_names[pred.item()]

In [68]:
#Run test:
result = predict_image(
    "data/bicycle/bicycle_14.jpg",
    model,
    transform,
    CLASS_NAMES
)

print("Prediction:", result)

bicycle 0.8478111624717712
car 0.0692635253071785
pedestrian 0.032883740961551666
truck 0.05004161223769188
Prediction: bicycle


In [67]:
#Test on whole dataset (accuracy check)
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 96.96969696969697
